# STEP 5, 6 & 7 - Baseline Selection & Parameter Derivation

## Objective
1.  **Select Neutral Baseline**: Choose the mode with the most balanced/stable profile from Step 3.
2.  **Derive Parameters**: Lock initial PCG settings based on this baseline's stats.
3.  **Transition**: Explicitly declare the end of calibration and start of training.


## Quick Start
Run the cell below to install dependencies.


In [ ]:
# %pip install pandas numpy


In [ ]:
import pandas as pd
import numpy as np
import os
import json

PROFILE_FILE = os.path.join('data', 'processed', 'mode_profiles.csv')
OUTPUT_PARAMS = os.path.join('config', 'initial_parameters.json')

if os.path.exists(PROFILE_FILE):
    df_profiles = pd.read_csv(PROFILE_FILE)
    print(f"Loaded Profiles for {df_profiles['modeId'].nunique()} modes.")
else:
    print("ERROR: Profiles not found. Run Notebook 02 first.")


## Step 5: Neutral Baseline Selection
**Criteria**:
1.  **Balanced Activity**: Presence of Combat, Exploration, and Collection (Low Sparsity across all dimensions).
2.  **Stability**: Lowest weighted variance.
3.  **Safety**: Death rate closest to 0 without being 0 (non-trivial).

We score each mode to suggest a candidate.


In [ ]:
def score_baseline(df_profiles):
    """Multi-objective baseline ranking using z-score normalised criteria.
    
    Replaces the previous heuristic (sparsity*1 + deaths*100 + std*0.1) with an
    explicit z-score aggregation so weights are interpretable and unit-invariant.
    Lower neutrality_score = more neutral (better baseline candidate).
    """
    modes = df_profiles['modeId'].unique()
    criteria = []

    for mode in modes:
        stats = df_profiles[df_profiles['modeId'] == mode]

        mean_sparsity = stats['sparsity_pct'].mean()
        mean_std = stats['std'].mean()

        death_row = stats[stats['metric'] == 'deathCountInWindow']
        death_rate = death_row.iloc[0]['mean'] if not death_row.empty else 0

        criteria.append({
            'modeId': mode,
            'mean_sparsity': mean_sparsity,
            'mean_std': mean_std,
            'death_rate': death_rate,
        })

    crit_df = pd.DataFrame(criteria)

    # Z-score each criterion across modes (ddof=0: population std, safe for small N)
    for col in ['mean_sparsity', 'mean_std', 'death_rate']:
        mu = crit_df[col].mean()
        sigma = crit_df[col].std(ddof=0)
        crit_df[f'z_{col}'] = (crit_df[col] - mu) / sigma if sigma > 0 else 0.0

    # Explicit weights with documented rationale:
    #   mean_sparsity 0.4 -- primary neutrality signal; a balanced mode exercises all mechanics
    #   mean_std      0.3 -- stability; high variance indicates chaotic or degenerate runs
    #   death_rate    0.3 -- safety; non-zero but low is ideal; heavily penalised above 0.5/window
    WEIGHTS = {'z_mean_sparsity': 0.4, 'z_mean_std': 0.3, 'z_death_rate': 0.3}

    crit_df['neutrality_score'] = (
        WEIGHTS['z_mean_sparsity'] * crit_df['z_mean_sparsity'] +
        WEIGHTS['z_mean_std']      * crit_df['z_mean_std']      +
        WEIGHTS['z_death_rate']    * crit_df['z_death_rate']
    )

    return crit_df[['modeId', 'mean_sparsity', 'mean_std', 'death_rate', 'neutrality_score']].sort_values('neutrality_score').reset_index(drop=True)


baseline_scores = score_baseline(df_profiles)
print('Baseline Candidate Ranking (Lower NeutralityScore = More Neutral):')
print(baseline_scores)

selected_mode = baseline_scores.iloc[0]['modeId']
print(f'SELECTED BASELINE: {selected_mode}')


## Step 6: Parameter Finalisation
We derive the initial parameters from the Selected Baseline's mean values.
Example Derivation: `InitialEnemyDensity = BaselineMean_EnemiesHit * 1.5` (Margin for challenge)


In [ ]:
baseline_stats = df_profiles[df_profiles['modeId'] == selected_mode]
derived_params = {
    "_meta": {
        "source_mode": selected_mode,
        "derivation_method": "Mean + 20% Margin"
    }
}

for idx, row in baseline_stats.iterrows():
    metric = row['metric']
    val = row['mean']
    
    # Example derivation logic
    derived_params[f"target_{metric}"] = val * 1.2

# Ensure directory exists
os.makedirs('config', exist_ok=True)

with open(OUTPUT_PARAMS, 'w') as f:
    json.dump(derived_params, f, indent=2)

print(f"Final Parameters Saved to: {OUTPUT_PARAMS}")
print(json.dumps(derived_params, indent=2))


In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans

mode1_data = df_profiles[df_profiles['modeId'] == selected_mode][['mean']].values

scaler = MinMaxScaler().fit(mode1_data)
scaler_params = {'data_min_': scaler.data_min_.tolist(), 'data_max_': scaler.data_max_.tolist()}
with open(os.path.join('config', 'scaler_params.json'), 'w') as f:
    json.dump(scaler_params, f, indent=2)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10).fit(mode1_data)
centroids = {'cluster_centers_': kmeans.cluster_centers_.tolist()}
with open(os.path.join('config', 'centroids.json'), 'w') as f:
    json.dump(centroids, f, indent=2)
print(f"scaler_params.json | min={scaler.data_min_} max={scaler.data_max_}")
print(f"centroids.json | centers={kmeans.cluster_centers_.flatten().round(3)}")
print(f"initial_parameters.json saved to: {OUTPUT_PARAMS}")

## Step 7: Explicit Transition

> **DECLARATION**
> 
> The Calibration Phase is hereby **COMPLETE**.
> 
> *   Calibration Data (`calibration_dataset.csv`) is **FROZEN** and used ONLY for the derivations above.
> *   Initial Training Parameters are **LOCKED** in `config/initial_parameters.json`.
> *   Future Telemetry will be strictly for **MODEL TRAINING** (Adaptation).
> *   No feedback loop exists between Training Telemetry and these Calibration Parameters.
